In [15]:
import os

In [16]:
os.environ["MLFLOW_TRACKING_URL"] = "https://dagshub.com/Temmytope-seun/dscProject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Temmytope-seun"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "ffd4eb226da5bd1070fe9492741f8ef8728c0623"

In [17]:
%pwd

'c:\\Users\\temmy\\OneDrive\\Desktop\\mlUdemy\\dscProject'

In [26]:
os.chdir('c:/Users/temmy/OneDrive/Desktop/mlUdemy/dscProject')
%pwd

'c:\\Users\\temmy\\OneDrive\\Desktop\\mlUdemy\\dscProject'

In [19]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metrics_file_path: Path
    target_column: str
    mlflow_uri: str

In [20]:
from src.dscProject.constants import *
from src.dscProject.utils.common import read_yaml, create_directories, save_json

In [21]:
class ConfigurationManager:
    def __init__(self,
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH,
                schema_filepath=SCHEMA_FILE_PATH
                ):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metrics_file_path=config.metrics_file_path,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/Temmytope-seun/dscProject.mlflow"
        )
        return model_evaluation_config

In [ ]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
from urllib.parse import urlparse
import numpy as np

In [ ]:
import mlflow.sklearn

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        # Calculate evaluation metrics
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        # Load the test data
        test_data = pd.read_csv(self.config.test_data_path)
        # Load the trained model
        model = joblib.load(self.config.model_path)
        # Separate features and target variable
        X_test = test_data.drop([self.config.target_column], axis=1)
        y_test = test_data[self.config.target_column]
        
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        with mlflow.start_run():
            # Make predictions
            y_pred = model.predict(X_test)
            (rmse, mae, r2) = self.eval_metrics(y_test, y_pred)

            # Save metrics to a JSON file
            metrics = {
                "RMSE": rmse,
                "MAE": mae,
                "R2_Score": r2
            }
            save_json(path=Path(self.config.metrics_file_path), data=metrics)
                    
            # Log metrics to MLflow
            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("RMSE", rmse)
            mlflow.log_metric("MAE", mae)
            mlflow.log_metric("R2_Score", r2)
            
            # mlflow.sklearn.log_model(model, "model")
            # run_id = run.info.run_id

        # mlflow.register_model(
        #     f"runs:/{run_id}/model",
        #     "ElasticNetModel"
        # )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel") # type: ignore
            else:
                mlflow.sklearn.log_model(model, "model") # type: ignore

        

In [38]:
try:
    config = ConfigurationManager()
    model_eval_config = config.get_model_evaluation_config()
    model_eval = ModelEvaluation(config=model_eval_config)
    model_eval.log_into_mlflow()
except Exception as e:
    raise e

[2026-06-04 19:29:19,832: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-04 19:29:19,864: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-04 19:29:19,871: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-04 19:29:19,881: INFO: common: created directory at: artifacts_root]
[2026-06-04 19:29:19,888: INFO: common: created directory at: artifacts/model_evaluation]
trackinggggg https
https://dagshub.com/Temmytope-seun/dscProject.mlflow
3.12.0
[2026-06-04 19:29:21,140: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/06/04 19:29:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 19:29:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticNetModel' already exists. Creating a new version of this model...
2026/06/04 19:29:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.


🏃 View run likeable-horse-546 at: https://dagshub.com/Temmytope-seun/dscProject.mlflow/#/experiments/0/runs/5f63dba5702742f4929f3e28f675f97d
🧪 View experiment at: https://dagshub.com/Temmytope-seun/dscProject.mlflow/#/experiments/0
